week 3

In [60]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
import statsmodels.api as sm

In [61]:
# Load datasets
austin_df = pd.read_csv("/Users/sohanaziz/Documents/AI for leaders sem 2/Datasets/cleaned datasets/week 2/cleaned_austin_data.csv")
airbnb_df = pd.read_csv("/Users/sohanaziz/Documents/AI for leaders sem 2/Datasets/cleaned datasets/week 2/listings_cleaned.csv")


### Austin dataset

In [62]:

# Define preprocessing function
def preprocess_data(df, target_column):
    df = df.dropna()
    df_numeric = df.select_dtypes(include=[np.number])
    y = df_numeric[target_column]
    X = df_numeric.drop(columns=[target_column])
    return train_test_split(X, y, test_size=0.2, random_state=42)

# Austin: predict latestPrice
X_train_austin, X_test_austin, y_train_austin, y_test_austin = preprocess_data(austin_df, 'latestPrice')

# Airbnb: predict price
X_train_airbnb, X_test_airbnb, y_train_airbnb, y_test_airbnb = preprocess_data(airbnb_df, 'price')

###  Linear Regression - Forward Selection

In [63]:
def forward_selection(X, y):
    remaining = list(X.columns)
    selected = []
    current_score, best_new_score = float('inf'), float('inf')

    while remaining:
        scores_with_candidates = []
        for candidate in remaining:
            model = sm.OLS(y, sm.add_constant(X[selected + [candidate]])).fit()
            aic = model.aic
            scores_with_candidates.append((aic, candidate))
        scores_with_candidates.sort()
        best_new_score, best_candidate = scores_with_candidates[0]
        if best_new_score < current_score:
            remaining.remove(best_candidate)
            selected.append(best_candidate)
            current_score = best_new_score
        else:
            break
    return selected


In [64]:
# Austin
features_fwd_austin = forward_selection(X_train_austin, y_train_austin)
print("Austin Forward Selection Features:", features_fwd_austin)


/Users/sohanaziz/opt/anaconda3/lib/python3.9/site-packages/statsmodels/tsa/tsatools.py:142: FutureWarning: In a future version of pandas all arguments of concat except for the argument 'objs' will be keyword-only
  x = pd.concat(x[::order], 1)


Austin Forward Selection Features: ['numOfBathrooms', 'livingAreaSqFt', 'yearBuilt', 'zipcode', 'homeType_Vacant Land', 'propertyTaxRate', 'avgSchoolRating', 'numOfWaterfrontFeatures', 'numOfStories', 'avgSchoolSize', 'numOfPatioAndPorchFeatures', 'numOfElementarySchools', 'numPriceChanges', 'numOfParkingFeatures', 'numOfBedrooms', 'numOfMiddleSchools', 'homeType_Multiple Occupancy', 'numOfHighSchools', 'MedianStudentsPerTeacher', 'numOfPrimarySchools', 'avgSchoolDistance', 'numOfCommunityFeatures', 'homeType_Residential', 'homeType_Condo', 'homeType_Townhouse', 'homeType_Single Family', 'homeType_MultiFamily', 'homeType_Mobile / Manufactured', 'numOfWindowFeatures', 'lotSizeSqFt', 'numOfAccessibilityFeatures']


In [65]:
# Airbnb
features_fwd_airbnb = forward_selection(X_train_airbnb, y_train_airbnb)
print("Airbnb Forward Selection Features:", features_fwd_airbnb)

Airbnb Forward Selection Features: ['host_total_listings_count', 'host_listings_count', 'room_type_Hotel room', 'bathrooms', 'reviews_per_month', 'availability_30', 'maximum_minimum_nights', 'minimum_nights_avg_ntm', 'minimum_nights', 'amenity_wifi', 'amenity_air_conditioning', 'latitude', 'host_id', 'availability_365', 'calculated_host_listings_count', 'room_type_Private room', 'host_tenure_years', 'neighbourhood_cleansed', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_accuracy', 'minimum_minimum_nights', 'host_identity_verified', 'availability_60', 'review_scores_rating', 'host_is_superhost', 'accommodates', 'calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms', 'calculated_host_listings_count_entire_homes', 'maximum_nights', 'host_has_profile_pic', 'review_scores_communication', 'amenity_heating', 'review_scores_location']


### Linear Regressiion- Backward Elimination

In [66]:
def backward_elimination(X, y, threshold=0.05):
    features = list(X.columns)
    while True:
        X_const = sm.add_constant(X[features])
        model = sm.OLS(y, X_const).fit()
        pvalues = model.pvalues.iloc[1:]  # Exclude intercept
        max_pval = pvalues.max()
        if max_pval > threshold:
            excluded = pvalues.idxmax()
            features.remove(excluded)
        else:
            break
    return features

In [67]:
# Austin
features_bwd_austin = backward_elimination(X_train_austin, y_train_austin)
print("Austin Backward Elimination Features:", features_bwd_austin)


Austin Backward Elimination Features: ['zipcode', 'propertyTaxRate', 'yearBuilt', 'numPriceChanges', 'numOfParkingFeatures', 'numOfPatioAndPorchFeatures', 'numOfWaterfrontFeatures', 'numOfCommunityFeatures', 'livingAreaSqFt', 'numOfPrimarySchools', 'numOfElementarySchools', 'numOfMiddleSchools', 'numOfHighSchools', 'avgSchoolDistance', 'avgSchoolRating', 'avgSchoolSize', 'MedianStudentsPerTeacher', 'numOfBathrooms', 'numOfBedrooms', 'numOfStories', 'homeType_Condo', 'homeType_Mobile / Manufactured', 'homeType_MultiFamily', 'homeType_Multiple Occupancy', 'homeType_Single Family', 'homeType_Townhouse', 'homeType_Vacant Land']


/Users/sohanaziz/opt/anaconda3/lib/python3.9/site-packages/statsmodels/tsa/tsatools.py:142: FutureWarning: In a future version of pandas all arguments of concat except for the argument 'objs' will be keyword-only
  x = pd.concat(x[::order], 1)
/Users/sohanaziz/opt/anaconda3/lib/python3.9/site-packages/statsmodels/tsa/tsatools.py:142: FutureWarning: In a future version of pandas all arguments of concat except for the argument 'objs' will be keyword-only
  x = pd.concat(x[::order], 1)
/Users/sohanaziz/opt/anaconda3/lib/python3.9/site-packages/statsmodels/tsa/tsatools.py:142: FutureWarning: In a future version of pandas all arguments of concat except for the argument 'objs' will be keyword-only
  x = pd.concat(x[::order], 1)
/Users/sohanaziz/opt/anaconda3/lib/python3.9/site-packages/statsmodels/tsa/tsatools.py:142: FutureWarning: In a future version of pandas all arguments of concat except for the argument 'objs' will be keyword-only
  x = pd.concat(x[::order], 1)
/Users/sohanaziz/opt/ana

In [68]:
# Airbnb
features_bwd_airbnb = backward_elimination(X_train_airbnb, y_train_airbnb)
print("Airbnb Backward Elimination Features:", features_bwd_airbnb)

/Users/sohanaziz/opt/anaconda3/lib/python3.9/site-packages/statsmodels/tsa/tsatools.py:142: FutureWarning: In a future version of pandas all arguments of concat except for the argument 'objs' will be keyword-only
  x = pd.concat(x[::order], 1)
/Users/sohanaziz/opt/anaconda3/lib/python3.9/site-packages/statsmodels/tsa/tsatools.py:142: FutureWarning: In a future version of pandas all arguments of concat except for the argument 'objs' will be keyword-only
  x = pd.concat(x[::order], 1)
/Users/sohanaziz/opt/anaconda3/lib/python3.9/site-packages/statsmodels/tsa/tsatools.py:142: FutureWarning: In a future version of pandas all arguments of concat except for the argument 'objs' will be keyword-only
  x = pd.concat(x[::order], 1)
/Users/sohanaziz/opt/anaconda3/lib/python3.9/site-packages/statsmodels/tsa/tsatools.py:142: FutureWarning: In a future version of pandas all arguments of concat except for the argument 'objs' will be keyword-only
  x = pd.concat(x[::order], 1)


Airbnb Backward Elimination Features: ['id', 'host_id', 'host_is_superhost', 'host_listings_count', 'host_total_listings_count', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood_cleansed', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_nights_avg_ntm', 'has_availability', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'instant_bookable', 'calculated_host_listings_count', 'calculated_host_listings_count_entire_homes', 'calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms', 'reviews_per_month', 'num_amenities', 'amenity_wifi', 'ame

### PCR

In [69]:
def run_pcr(X, y, max_components=20):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA()
    X_pca = pca.fit_transform(X_scaled)

    scores = []
    for i in range(1, min(max_components, X.shape[1]) + 1):
        score = cross_val_score(LinearRegression(), X_pca[:, :i], y, scoring='neg_mean_squared_error', cv=5).mean()
        scores.append(score)

    best_n = np.argmax(scores) + 1
    print(f"Optimal PCR components: {best_n}")
    return best_n, scores

# Austin
print("Running PCR for Austin housing dataset...")
pcr_n_austin, _ = run_pcr(X_train_austin, y_train_austin)

# Airbnb
print("Running PCR for Airbnb listings dataset...")
pcr_n_airbnb, _ = run_pcr(X_train_airbnb, y_train_airbnb)

Running PCR for Austin housing dataset...
Optimal PCR components: 12
Running PCR for Airbnb listings dataset...
Optimal PCR components: 20


### PLSR

In [70]:
def run_plsr(X, y, max_components=20):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    scores = []
    for i in range(1, min(max_components, X.shape[1]) + 1):
        pls = PLSRegression(n_components=i)
        score = cross_val_score(pls, X_scaled, y, scoring='neg_mean_squared_error', cv=5).mean()
        scores.append(score)

    best_n = np.argmax(scores) + 1
    print(f"Optimal PLS components: {best_n}")
    return best_n, scores

# Austin
print("Running PLSR for Austin housing dataset...")
pls_n_austin, _ = run_plsr(X_train_austin, y_train_austin)

# Airbnb
print("Running PLSR for Airbnb listings dataset...")
pls_n_airbnb, _ = run_plsr(X_train_airbnb, y_train_airbnb)

Running PLSR for Austin housing dataset...
Optimal PLS components: 1
Running PLSR for Airbnb listings dataset...
Optimal PLS components: 20
